# Chapter 6: AI-Powered Design & Architecture

This notebook is the runnable companion to Chapter 6 of the SDLC book. It follows the **GlobalBank Corporate Onboarding** case study.

Here, we use the standard `openai` SDK to explore how AI accelerates software architecture: tradeoff analysis, ADR generation, automated diagramming, API contract design, and data modeling.

### Prerequisites
Ensure you have the `openai` package installed, and your OpenAI API key ready.

In [ ]:
!pip install -q openai

import os
import getpass
from openai import OpenAI

# Set your OpenAI API Key
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()

def generate_response(system_prompt, user_prompt):
    """Helper to call OpenAI and return the text"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

## 1. Architecture Exploration & Trade-off Analysis (§6.1)

AI drastically expands the architecture decision space. Instead of relying solely on the patterns we already know, we can ask AI to analyze our system constraints and present multiple viable topologies with an objective trade-off matrix.

In [ ]:
with open("data/system_context.txt", "r") as f:
    system_context = f.read()

system_prompt = """
You are a Principal Cloud Architect. I will provide a system context.
Propose 3 distinct architecture topologies (e.g. Serverless, Event-Driven Microservices, Modular Monolith) that could solve this problem.

For each, provide:
1. A brief summary of the approach.
2. A comparison matrix scoring them 1-10 on:
   - Scalability
   - Operational Complexity
   - Time to Market
   - Cost at Scale
   - Regulatory/Audit Compliance Ease
"""

architecture_options = generate_response(system_prompt, f"System Context:\n{system_context}")
print(architecture_options)

## 2. Design Document Generation (ADR) (§6.2)

Architecture Decision Records (ADRs) capture the *"why"* behind technical choices. By feeding AI a pull request description or a chat transcript, it can automatically formalize the discussion into a standard ADR format.

In [ ]:
with open("data/event_broker_pr.txt", "r") as f:
    pr_discussion = f.read()

system_prompt = """
You are a Staff Engineer. Read the provided Pull Request discussion about an architectural choice.
Generate an Architecture Decision Record (ADR) based on this discussion.

Use Michael Nygard's standard ADR format:
- Title (ADR-NNN: descriptive title)
- Status (Proposed)
- Context (What is the technical/business driver?)
- Decision (What are we choosing? Make a definitive recommendation based on the risks in the PR)
- Consequences (Positive, Negative, and Risks of this choice)
"""

adr_output = generate_response(system_prompt, f"PR Discussion:\n{pr_discussion}")
print(adr_output)

## 3. AI-Assisted Diagramming (C4 Model) (§6.3)

Diagrams usually go stale the moment they are drawn. By generating diagrams-as-code (like Mermaid or PlantUML) using AI, we can build pipelines that keep our architecture documentation living and up-to-date.

In [ ]:
system_prompt = """
You are a Software Architect who specializes in the C4 Modeling framework.
I will provide a system context. Generate a C4 Level 2 (Container) Diagram using Mermaid syntax (`mermaid`).

The diagram should show the users, the frontend SPA, the Backend API, the Database, the Event Broker, and External APIs (World-Check AML).
Only output the valid Mermaid code block so it renders correctly.
"""

diagram_mermaid = generate_response(system_prompt, f"System Context:\n{system_context}")
print(diagram_mermaid)

## 4. API Design & Contract-First Development (§6.4)

Writing OpenAPI boilerplate is tedious. AI accelerates "Contract-First" API design by quickly translating business requirements into a fully mature, RFC-compliant OpenAPI 3.1 specification.

In [ ]:
with open("data/api_requirements.txt", "r") as f:
    api_reqs = f.read()

system_prompt = """
You are an API Designer. Generate an OpenAPI 3.1 YAML specification based on the requirements.

Ensure the API demonstrates REST maturity:
- Use standard HTTP status codes (200, 201, 202, 400, 401, 404, 429)
- Document the `X-Correlation-ID` header.
- Include standard error responses using the RFC 7807 Problem Details format.
- Define the JWT Bearer security scheme.
"""

openapi_spec = generate_response(system_prompt, f"API Requirements:\n{api_reqs}")
print(openapi_spec)

## 5. Data Modeling & Schema Generation (§6.5)

Data modeling is foundational. AI can read entity requirements, design a 3rd Normal Form (3NF) relational schema, identify the correct indexes, and write the initial database migration script.

In [ ]:
with open("data/data_model_reqs.txt", "r") as f:
    db_reqs = f.read()

system_prompt = """
You are a Database Administrator tuning PostgreSQL.
Read the provided entity requirements for the GlobalBank IDP service.

Provide:
1. A Mermaid Entity-Relationship (ER) diagram representing the schema.
2. A list of 3-4 recommended B-Tree or Partial indexes to improve query performance on these tables, with your rationale.
3. The raw PostgreSQL `CREATE TABLE` script implementing this model.
"""

db_model = generate_response(system_prompt, f"Data Modeling Requirements:\n{db_reqs}")
print(db_model)